# Full Production Simulation & Analysis Pipeline Notebook

This notebook mirrors the production pipeline script `scripts/run_full_production.sh` 1-to-1.
It executes the complete 200 MeV electron injector linac optimization campaign across:
- **Step 1**: Setup Environment and Paths
- **Step 2**: Environment & Executable Verification
- **Step 3**: Phase 1 Scalarized Bayesian Optimization (`SingleTaskGP` / `qLogNEI`)
- **Step 4**: Phase 2 Unconstrained Multi-Objective Bayesian Optimization (`qLogNEHVI`)
- **Step 5**: Phase 3 Constraint-Aware Multi-Objective Bayesian Optimization (`qLogNEHVI`)
- **Step 6**: 3-Phase Comparative Analysis (Phase 1 vs Phase 2 vs Phase 3) & Pareto Audit
- **Step 7**: Engineering Tolerance Robustness Analysis
- **Step 8**: Final Summary & Verification Report Summary

In [ ]:
# Step 1 & 2: Setup Environment and Paths
import os
from pathlib import Path
import sys
import torch

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.environ["ASTRA_BIN"] = str(project_root / "bin" / "astra")
os.environ["GENERATOR_BIN"] = str(project_root / "bin" / "generator")

print(f"Project Root: {project_root}")
print(f"ASTRA Binary: {os.environ[ASTRA_BIN]}")

In [ ]:
# Step 3: Run Phase 1 Scalarized BO Simulation
from mobo_linac.cli import run_scalarized
import argparse

dir_p1 = project_root / "results" / "full_production" / "phase1_scalarized"
args_p1 = argparse.Namespace(
    config=str(project_root / "configs" / "mobo_200MeV.yaml"),
    n_iterations=5,
    batch_size=4,
    num_initial_samples=16,
    num_workers=4,
    weights=[1.0, 1.0, 1.0],
    seed=42,
    output_dir=str(dir_p1),
    device="auto",
)
print("Starting Phase 1 Scalarized BO...")
run_scalarized(args_p1)

In [ ]:
# Step 4: Run Phase 2 Unconstrained MOBO Simulation
from mobo_linac.campaigns.runner import MoboCampaignRunner

dir_p2 = project_root / "results" / "full_production" / "phase2_unconstrained"
runner_p2 = MoboCampaignRunner(
    config=project_root / "configs" / "mobo_200MeV.yaml",
    run_name="phase2_unconstrained",
    output_dir=dir_p2,
    num_initial_samples=16,
    num_batches=5,
    batch_size=4,
    num_workers=4,
    seed=42,
    acq_type="qLogNEHVI",
    constrained=False,
    device="auto",
)
res_p2, tracker_p2, _ = runner_p2.run()

In [ ]:
# Step 5: Run Phase 3 Constraint-Aware MOBO Simulation
dir_p3 = project_root / "results" / "full_production" / "phase3_constrained"
runner_p3 = MoboCampaignRunner(
    config=project_root / "configs" / "mobo_200MeV.yaml",
    run_name="phase3_constrained",
    output_dir=dir_p3,
    num_initial_samples=16,
    num_batches=5,
    batch_size=4,
    num_workers=4,
    seed=42,
    acq_type="qLogNEHVI",
    constrained=True,
    device="auto",
)
res_p3, tracker_p3, _ = runner_p3.run()

In [ ]:
# Step 6: 3-Phase Comparative Analysis (Phases 1, 2, and 3) & Pareto Verification Audit
from scripts.run_comparison_and_verification import generate_three_phase_report, load_phase_results, verify_pareto_candidates
from mobo_linac.config import load_config

cfg = load_config(project_root / "configs" / "mobo_200MeV.yaml")
analysis_dir = project_root / "results" / "full_production" / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

res_p1 = load_phase_results(dir_p1, cfg)
verif_records = verify_pareto_candidates(res_p3, cfg, analysis_dir)
report_file = generate_three_phase_report(
    p1_dir=dir_p1, res_p1=res_p1,
    p2_dir=dir_p2, res_p2=res_p2,
    p3_dir=dir_p3, res_p3=res_p3,
    verification_records=verif_records,
    output_dir=analysis_dir,
)
print(f"3-Phase comparative analysis report generated at: {report_file}")

In [ ]:
# Step 7: Engineering Tolerance Robustness Analysis
from scripts.run_robustness_analysis import run_robustness_analysis
import argparse

args_rob = argparse.Namespace(
    pareto_csv=str(dir_p3 / "pareto.csv"),
    config=str(project_root / "configs" / "mobo_200MeV.yaml"),
    output_dir=str(analysis_dir / "robustness"),
    num_workers=4,
    num_perturbations=10,
    seed=42,
)
run_robustness_analysis(args_rob)

In [ ]:
# Step 8: Final Summary & Verification Report Summary
report_path = analysis_dir / "comparison_report.md"
print(f"Summary Report Path: {report_path}")
if report_path.exists():
    print(report_path.read_text())